# Post-hoc attribution: exact source spans

Same frozen seven passages and editor draft settings. Qwen selects source-span IDs; Python slices the original text. This guarantees literal quotation provenance, not claim support. The previous post-hoc result is retained for comparison.


In [ ]:
SETTINGS = {
    "source_report": "reports/reranker-comparison-20261114T104507Z.json",
    "context_tokens": 8192,
    "target_words": 500,
    "article_tokens": 4000,
    "article_seconds": 300,
    "reference_top_k": 3,
    "attribution_batch_size": 5,
    "attribution_tokens": 1200,
    "attribution_seconds": 90,
}
SETTINGS["attribution_mode"] = "source_spans"
SETTINGS.update(citation_unit="sentence", max_spans_per_unit=4)  # Historical experiment


In [ ]:
from pathlib import Path
import sys, json, html, hashlib, importlib
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display, HTML
project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wikilens").is_dir())
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
assert sys.version_info[:2] == (3, 12), "Use the project Python 3.12 kernel."
assert Path(sys.prefix).resolve() == (project_dir / ".venv").resolve(), "Select the project .venv kernel."
import wikilens.citation_pipelines as citation_module
citation_module = importlib.reload(citation_module)
source_path = project_dir / SETTINGS["source_report"]
capture = json.loads(source_path.read_text())
shared = capture["variants"]["cross_encoder"]["run"]
shared_nodes = shared["stages"]["postprocessing"]["nodes"]
results = {}
print("Question:",shared["question"],"| focus:",shared["search_focus"])
print("Qwen revision:",shared["runtime"]["revisions"]["qwen"])
display(pd.DataFrame([{"id":n["id"],"title":n["title"],"lang":n["lang"],"characters":len(n["text"])} for n in shared_nodes]))

def show_result(result):
    print("Status:",result["status"],"| elapsed:",round(result["seconds"],2),"seconds")
    previous = None
    blocks = []
    for sentence in result["sentences"]:
        if sentence["section"] != previous:
            blocks.append("<h3>"+html.escape(sentence["section"])+"</h3>")
            previous = sentence["section"]
        links = []
        for citation in sentence["citations"]:
            label = "["+citation["source_id"]+"]" + (" ?" if citation.get("quote") and not citation.get("quote_located") else "")
            if citation.get("revision_url"):
                links.append('<a target="_blank" rel="noopener" href="'+html.escape(citation["revision_url"],quote=True)+'">'+html.escape(label)+'</a>')
            else:
                links.append(html.escape(label))
        if sentence.get("citation_needed"):
            links.append('<span style="color:#9b5700">[citation needed]</span>')
        blocks.append("<p><small>"+html.escape(sentence["claim_id"])+"</small> "+html.escape(sentence["text"])+" "+" ".join(links)+"</p>")
    display(HTML('<article style="max-width:850px;line-height:1.8"><p>AI draft · citation support pending review</p>'+"".join(blocks)+"</article>"))
    display(pd.DataFrame([result["mechanical_metrics"]]))
    display(HTML("<details><summary>Raw model calls and evidence quotations</summary><pre>"+html.escape(json.dumps(result["calls"],ensure_ascii=False,indent=2))+"</pre></details>"))


## 3 · Draft first, add citations afterwards

One measured pipeline run. It owns one GPU model at a time and releases it on completion.


In [ ]:
results["post_hoc"] = citation_module.run_pipeline(capture, "post_hoc", SETTINGS)
show_result(results["post_hoc"])


In [ ]:
from pathlib import Path
import json, html
import pandas as pd
from IPython.display import display, HTML, Markdown
project_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wikilens").is_dir())
p = project_dir / "reports/exact-quotation-comparison-latest.json"
if not p.exists():
    print("Run scripts/compare-exact-quotations.py first.")
else:
    saved = json.loads(p.read_text())
    names = [name for name in saved.get("order", saved["variants"]) if name in saved["variants"]]
    display(pd.DataFrame([{"pipeline":name,"seconds":round(saved["variants"][name]["seconds"],2),
        **saved["variants"][name]["mechanical_metrics"],
        **(saved["variants"][name].get("citation_review") or {}).get("metrics",{})} for name in names]))
    display(Markdown("Citation accuracy measures full support among cited sentences; coverage uses all factual sentences. "
        "**Support + provenance** additionally requires a located quotation when the workflow requests one. "
        "Prose generation does not request quotations. An uncited sentence is not necessarily false. "
        "These are assistant judgments on one development example, not a benchmark or human review."))
    display(pd.DataFrame([{"pipeline":name,**saved["variants"][name]["stages"]} for name in names]).fillna("—"))
    for name in names:
        item = saved["variants"][name]
        review = item.get("citation_review") or {}
        display(Markdown("### "+name+"\n\n"+review.get("summary","Review pending")))
        blocks = []; section = None
        for sentence in item["sentences"]:
            if sentence["section"] != section:
                blocks.append("<h3>"+html.escape(sentence["section"])+"</h3>")
                section = sentence["section"]
            links = []; quotes = []
            for citation in sentence["citations"]:
                label = "["+citation["source_id"]+"]"
                url = citation.get("revision_url")
                links.append("<a target='_blank' rel='noopener' href='"+html.escape(url,quote=True)+"'>"+label+"</a>" if url else label)
                if citation.get("quote"):
                    quotes.append("<blockquote style='white-space:pre-wrap'>"+html.escape(citation["quote"])+"</blockquote><p>"+html.escape(citation.get("status","unchecked"))+"</p>")
            markers = " ".join(links) or "<em style='color:#996000'>[citation needed]</em>"
            blocks.append("<p><small>"+sentence["claim_id"]+"</small> "+html.escape(sentence["text"])+" "+markers+"</p>")
            if quotes:
                blocks.append("<details><summary>Requested supporting quotation(s)</summary>"+"".join(quotes)+"</details>")
        display(HTML("".join(blocks)))
        if review.get("claims"):
            display(pd.DataFrame(review["claims"]))
    if saved.get("experiment_history"):
        display(Markdown("Earlier structured attempts and failures remain preserved in: " + "; ".join(x["report"] for x in saved["experiment_history"])))
